In [ ]:
# === STANDART COLAB KURULUM HÜCRESİ ===
from google.colab import drive, userdata
import os, sys

drive.mount("/content/drive")

REPO_URL = "https://github.com/Oldzhay34/receipts-and-interactions.git"
BRANCH   = "b/1.2-quality"

try:
    TOKEN = userdata.get("GH_TOKEN")
except userdata.SecretNotFoundError:
    TOKEN = None

url = REPO_URL.replace("https://", f"https://{TOKEN}@") if TOKEN else REPO_URL

if not os.path.exists("/content/repo"):
    !git clone -q -b {BRANCH} {url} /content/repo
else:
    !cd /content/repo && git fetch -q && git checkout -q {BRANCH} && git pull -q

%cd /content/repo
!pip install -q -r requirements.txt && pip install -q -e .

os.environ["DATA_ROOT"] = "/content/drive/MyDrive/CME4403/data"
if "/content/repo/src" not in sys.path:
    sys.path.insert(0, "/content/repo/src")
from recsys.config import load_config
CFG = load_config()
print("commit:", os.popen("git rev-parse --short HEAD").read().strip())

# 02 — Data Quality Report

Stage 1.2. This notebook contains no business logic; it only calls `recsys.analysis.quality`.

* Input: stage **`loaded`** (LATEST), validated against the loaded schemas.
* Output: one **`quality`** version with the DQR tables (continuous, categorical, list columns), the issues table (validated against `QUALITY_ISSUES_SCHEMA`, read by 1.3 and 1.4) and PNG renderings for the report.
* `example_ids` holds recipe ids for recipe rules and `user_id:recipe_id` for interaction rules.

In [ ]:
from recsys.analysis.quality import (
    categorical_dqr,
    continuous_dqr,
    detect_issues,
    list_column_dqr,
    render_table_png,
)
from recsys.contracts import (
    LOADED_INTERACTIONS_SCHEMA,
    LOADED_RECIPES_SCHEMA,
    NUTRITION_COLUMNS,
    QUALITY_ISSUES_SCHEMA,
    validate,
)
from recsys.io import ArtifactStore

store = ArtifactStore.from_config(CFG)
loaded = store.load("loaded")
recipes = validate(loaded.objects["recipes"], LOADED_RECIPES_SCHEMA)
interactions = validate(loaded.objects["interactions"], LOADED_INTERACTIONS_SCHEMA)
print("input: loaded", loaded.version, "| recipes", recipes.shape, "| interactions", interactions.shape)

cont_cols = ["minutes", "n_steps", "n_ingredients", *NUTRITION_COLUMNS]
dqr_continuous = continuous_dqr(recipes, cont_cols)
dqr_categorical = categorical_dqr(interactions, ["rating"])
dqr_lists = list_column_dqr(recipes, ["ingredients", "tags"])

issues = detect_issues(recipes, interactions)
validate(issues, QUALITY_ISSUES_SCHEMA)

display(dqr_continuous)
display(dqr_categorical)
display(dqr_lists)
display(issues)

## Save stage `quality`

In [ ]:
import matplotlib.pyplot as plt

fig_cont = render_table_png(dqr_continuous, "Continuous Features DQR")
fig_cat = render_table_png(dqr_categorical, "Categorical Features DQR")
fig_list = render_table_png(dqr_lists, "List Columns DQR")
fig_issues = render_table_png(issues, "Data Quality Issues Registry")

saved = store.save(
    stage="quality",
    objects={
        "dqr_continuous": dqr_continuous,
        "dqr_categorical": dqr_categorical,
        "dqr_lists": dqr_lists,
        "issues": issues,
        "dqr_continuous_table": fig_cont,
        "dqr_categorical_table": fig_cat,
        "dqr_lists_table": fig_list,
        "issues_table": fig_issues,
    },
    inputs=[loaded],
    params={"source_stage": "loaded"},
    schemas={"issues": QUALITY_ISSUES_SCHEMA},
)

plt.close("all")
print("saved:", saved.stage, saved.version, "->", saved.path)
check = store.load("quality")  # re-read with sha256 verification
{name: f.get("n_rows") for name, f in check.manifest["files"].items()}